# Breakout Room #3 — Recomendar: filtrado colaborativo por ítem y por cliente

**Objetivo:**
- Recomendar productos con la matriz del breakout anterior, de dos maneras: **por ítem** y **por cliente**.
- Entender *por qué* cada método recomienda lo que recomienda.
- Comparar costo, estabilidad y qué pasa con clientes sin historia.

Dos ideas, la misma matriz:

| | Pregunta que contesta | Cómo puntúa un producto candidato |
|---|---|---|
| **Por ítem** (item-based) | "A quien compró X, ¿qué otros productos *se parecen* a lo que ya compró?" | suma de la similitud entre el candidato y **cada producto que el cliente ya tiene** |
| **Por cliente** (user-based) | "¿Qué compraron los clientes *parecidos a mí* que yo no he comprado?" | suma de lo que compraron los **K vecinos más parecidos**, ponderada por su similitud |

Donde dice **`# TODO`** hay algo para que ustedes completen.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
VERDE, OSCURO, NARANJA, GRIS = "#00704A", "#11241C", "#C4512F", "#9A988E"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

orders = pd.read_csv("data/orders.csv", parse_dates=["order_date"])
customers = pd.read_csv("data/customers.csv")
CUTOFF = orders["order_date"].max() - pd.Timedelta(days=90)
train = orders[orders["order_date"] <= CUTOFF]

# La matriz del breakout anterior (1 = el cliente compró ese producto antes del corte)
M = (pd.crosstab(train["customer_id"], train["item"]) > 0).astype(int).reindex(customers["customer_id"], fill_value=0)
X = M.values.astype(float)
productos = list(M.columns)
print("Matriz:", M.shape)

## Paso 1 — las similitudes

Dos matrices de similitud coseno: **productos × productos** (29 × 29) y **clientes × clientes** (2,000 × 2,000).
Ponemos la diagonal en 0 (un producto o cliente no es "vecino" de sí mismo). Fíjense en cuánto tarda cada una.

In [ ]:
t0 = time.perf_counter(); S_item = cosine_similarity(X.T); np.fill_diagonal(S_item, 0); t_item = time.perf_counter() - t0
t0 = time.perf_counter(); S_user = cosine_similarity(X);   np.fill_diagonal(S_user, 0); t_user = time.perf_counter() - t0
print(f"Matriz producto×producto: {S_item.shape}  → {S_item.size:,} celdas · {t_item*1000:.1f} ms")
print(f"Matriz cliente×cliente:   {S_user.shape}  → {S_user.size:,} celdas · {t_user*1000:.1f} ms")

## Paso 2 — recomendar por ítem

Para un cliente, el puntaje de cada producto es la suma de su similitud con **los productos que el cliente ya compró**.
Ya excluimos lo que el cliente tiene (no se recomienda lo que ya compró) y el **Agua** (ya vimos que no distingue a nadie).

In [ ]:
def top_k(scores, cid, k=3):
    owned = M.loc[cid].values > 0
    s = pd.Series(scores, index=productos).where(~owned).drop("Agua", errors="ignore")
    return s.sort_values(ascending=False).head(k)

def puntaje_item(cid):
    u = M.loc[cid].values.astype(float)         # lo que el cliente compró (vector de 0/1)
    # TODO 1: puntaje de cada producto = suma de su similitud con lo que el cliente ya compró.
    # Pista: es un producto punto (matricial) entre u y S_item.
    return ...                                  # <-- reemplacen esto

for cid in (14, 62):
    print(f"Cliente {cid} compró: {', '.join(M.columns[M.loc[cid] == 1])}")
    print(top_k(puntaje_item(cid), cid).round(2).to_string(), "\n")

## Paso 3 — recomendar por cliente

Buscamos a los **K = 30 clientes más parecidos** y sumamos lo que compraron, ponderado por cuánto se parecen.

In [ ]:
K_VEC = 30

def puntaje_user(cid):
    i = M.index.get_loc(cid)
    sims = S_user[i]                              # similitud de este cliente con todos los demás
    vecinos = np.argsort(-sims)[:K_VEC]           # los K más parecidos
    # TODO 2: puntaje de cada producto = suma de lo que compraron los vecinos, ponderada por su similitud.
    # Pista: sims[vecinos] (un vector de K pesos) por X[vecinos] (K filas de la matriz).
    return ...                                    # <-- reemplacen esto

for cid in (14, 62):
    print(f"Cliente {cid}:")
    print(top_k(puntaje_user(cid), cid).round(2).to_string(), "\n")

## Paso 4 — ¿por qué recomienda eso? (explicabilidad)

Un recomendador que no se puede explicar es difícil de defender ante el negocio. Para el cliente 14, miremos el **motivo** de la primera recomendación de cada método.

In [ ]:
cid = 14
rec_i = top_k(puntaje_item(cid), cid, 1).index[0]
rec_u = top_k(puntaje_user(cid), cid, 1).index[0]

# Por ítem: ¿de qué productos que ya compró viene el puntaje?
j = productos.index(rec_i)
tiene = [p for p in productos if M.loc[cid, p] == 1 and p != "Agua"]
motivo_i = pd.Series({p: S_item[productos.index(p), j] for p in tiene}).sort_values(ascending=False)
print(f"POR ÍTEM → '{rec_i}' porque se parece a lo que ya compró:")
print(motivo_i.round(2).head(4).to_string(), "\n")

# Por cliente: ¿qué vecinos lo compraron?
i = M.index.get_loc(cid)
vec = np.argsort(-S_user[i])[:K_VEC]
con_prod = [(int(M.index[v]), round(float(S_user[i, v]), 2)) for v in vec if M.iloc[v][rec_u] == 1][:4]
print(f"POR CLIENTE → '{rec_u}' porque lo compraron vecinos como (cliente, similitud): {con_prod}")
print(f"   {sum(M.iloc[v][rec_u] for v in vec)} de {K_VEC} vecinos lo compraron.")

## Paso 5 — ¿coinciden los dos métodos? ¿cuánto cuesta cada uno?

In [ ]:
con_hist = M.index[M.sum(axis=1) >= 2]
col_agua = productos.index("Agua")
def mejor(sc, owned):
    s = np.where(owned > 0, -np.inf, sc); s[col_agua] = -np.inf
    return np.argsort(-s)[:3]

coinc1, coinc3 = [], []
t0 = time.perf_counter()
for cid in con_hist:
    owned = M.loc[cid].values
    ti, tu = mejor(puntaje_item(cid), owned), mejor(puntaje_user(cid), owned)
    coinc1.append(ti[0] == tu[0]); coinc3.append(len(set(ti) & set(tu)))
dt = time.perf_counter() - t0
print(f"Clientes con al menos 2 productos: {len(con_hist):,}")
print(f"Misma recomendación #1: {np.mean(coinc1):.0%}")
print(f"Coinciden en al menos 2 de las 3 primeras: {np.mean(np.array(coinc3) >= 2):.0%}")
print(f"Tiempo de recomendar a todos (los dos métodos): {dt:.1f} s")

## Paso 6 — los casos difíciles

¿Qué le recomienda cada método a un cliente con **un solo producto** y a uno con **cero**?

In [ ]:
uno = M.index[M.sum(axis=1) == 1][0]
cero = M.index[M.sum(axis=1) == 0][0]
pop = pd.Series(X.sum(axis=0), index=productos)

print(f"Cliente {uno} (compró: {', '.join(M.columns[M.loc[uno] == 1])})")
print("  por ítem   :", list(top_k(puntaje_item(uno), uno).index))
print("  por cliente:", list(top_k(puntaje_user(uno), uno).index))
print(f"\nCliente {cero} (0 productos)")
print("  por ítem   :", list(top_k(puntaje_item(cero), cero).round(2).items()))
print("  por cliente:", list(top_k(puntaje_user(cero), cero).round(2).items()))
print("  (todo en 0: el método no tiene nada que ordenar)")
print("\nRespaldo común: lo más popular →", list(pop.drop("Agua").sort_values(ascending=False).head(3).index))

1. Los dos métodos, ¿recomiendan lo mismo? ¿En qué casos preferirían cada uno?
2. Tiempos y tamaños del Paso 1: si la base fuera de 2 millones de clientes y 10,000 productos, ¿cuál de las dos matrices de similitud podrían calcular y guardar de antemano?
3. ¿Qué hacen con el cliente de 0 productos? ¿Y con un producto *nuevo* que nadie ha comprado todavía?
4. ¿cómo sabrían cuál de los dos recomienda *mejor*?